In [1]:
import pandas as pd

In [3]:
url = "https://github.com/laxmimerit/IMDB-Movie-Reviews-Large-Dataset-50k/raw/refs/heads/master/train.xlsx"
data = pd.read_excel(url)
data =  data.sample(500, random_state=42)
data.head()

,Reviews,Sentiment
6868,I really think that people are taking the wron...,pos
24016,There was a stylish approach to this film on t...,pos
9668,My wife and I just finished watching Bûsu AKA ...,neg
13640,As with all environmentally aware films from t...,pos
14018,"With an absolutely amazing cast and crew, this...",neg


In [17]:
import torch
from collections import Counter

def tokenize(text):
    return text.lower().split()

counter = Counter(word for review in data["Reviews"] for word in tokenize(review))
most_common = counter.most_common(2000)
vocab = {word: idx + 2 for idx, (word, _) in enumerate(most_common)}
vocab["<PAD>"] = 0
vocab["<UNK>"] = 1

def encode_review(review):
    return torch.tensor([vocab.get(word, vocab["<UNK>"]) for word in tokenize(review)], dtype=torch.long)


In [18]:
from torch.utils.data import Dataset, DataLoader

class IMDBdataset(Dataset):
    def __init__(self, df, tokenizer, max_len):
        self.X = [encode_review(review) for review in df["Reviews"]]
        self.y = [torch.tensor([1.0 if label == "pos" else 0.0]) for label in df["Sentiment"]]

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

In [19]:
def collate_fn(batch):
    Xs, ys = zip(*batch)
    max_len = max(len(x) for x in Xs)
    Xs_padded = [torch.cat([X, torch.zeros(max_len - len(X), dtype=torch.long)]) for X in Xs]
    return torch.stack(Xs_padded), torch.stack(ys)

train_loader = DataLoader(IMDBdataset(data, vocab, 200), batch_size=32, shuffle=True, collate_fn=collate_fn)

In [20]:
import torch.nn as nn

class SentimentRNN(nn.Module):
    def __init__(self, vocab_size, embedding_dim=64, hidden_dim=128):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.rnn = nn.RNN(embedding_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        embedded = self.embedding(x)
        _, hidden = self.rnn(embedded)
        x = self.fc(hidden.squeeze(0))
        output = self.sigmoid(x)
        return output


In [21]:
model = SentimentRNN(len(vocab))
criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [22]:
for epoch in range(10):
    total_loss = 0
    for X, y in train_loader:
        y_pred = model(X)
        loss = criterion(y_pred, y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}, Loss: {total_loss/len(train_loader)}")

Epoch 1, Loss: 0.7119463458657265
Epoch 2, Loss: 0.6959225982427597
Epoch 3, Loss: 0.6944543421268463
Epoch 4, Loss: 0.6909795664250851
Epoch 5, Loss: 0.6892783455550671
Epoch 6, Loss: 0.6903060935437679
Epoch 7, Loss: 0.6891286186873913
Epoch 8, Loss: 0.6799790225923061
Epoch 9, Loss: 0.6881830915808678
Epoch 10, Loss: 0.6881341934204102


In [23]:
def predict_sentiment(review):
    model.eval()
    with torch.no_grad():
        x = encode_review(review)
        x = x[:100]
        if len(x) < 100:
            x = torch.cat([x, torch.zeros(100 - len(x), dtype=torch.long)])
        x = x.unsqueeze(0)
        y_pred = model(x)
        prob = y_pred.item()
        label = "pos" if prob > 0.5 else "neg"
        print(f"Review: {review}\nPredicted Sentiment: {label} (Probability: {prob:.4f})")

In [ ]:
predict_sentiment("This movie was fantastic! I loved it.")
predict_sentiment("This movie was terrible. I hated it.")

Review: This movie was fantastic! I loved it.
Predicted Sentiment: pos (Probability: 0.5370)
Review: I hated it.
Predicted Sentiment: pos (Probability: 0.5370)
